In [1]:
################ Import necessary libraries
import pandas as pd
import numpy as np
import glob

In [2]:
################ Import processed data
market_share = pd.read_excel("C:/Users/Lenovo/Desktop/Dissertaion/Code/MSc-Dissertation/final_cleaned_data.xlsx")

# Import the steel price data
steel_price_file = pd.read_excel(
    "C:/Users/Lenovo/Desktop/Dissertaion/China Data/Instruments/steel price index.xlsx",
    sheet_name='Sheet1')

# Convert the steel price data to a DataFrame
steel_price = pd.DataFrame(steel_price_file)

In [3]:
################ Construct differentiation IVs
# Key idea is to express the position of a brand in product characteristic space
# Define funciton to construct differentiation IVs

def construct_differentiation_ivs(market_share, threshold_std=1.0):
    """
    Construct IVs using positional indexing (6th col as mass, 7th as power)
    
    Parameters:
    -----------
    market_share : DataFrame
        Must have at least 7 columns with:
        - 6th column as mass
        - 7th column as power
    threshold_std : float
        Number of standard deviations for local difference threshold
    
    Returns:
    --------
    DataFrame with original data plus 6 new IV columns
    """
    
    # Create working copy
    df = market_share.copy()
    
    # Get columns by position
    mass_col = df.columns[6]
    power_col = df.columns[7]
    
    print(f"Using columns: {mass_col} as mass, {power_col} as power")
    
    # Calculate IVs
    
    ## Sum of rivals
    df['sum_rival_mass'] = df.groupby('market_id')[mass_col].transform('sum') - df[mass_col]
    df['sum_rival_power'] = df.groupby('market_id')[power_col].transform('sum') - df[power_col]
    
    ## Euclidean distance (vectorized)
    def euclidean(group, col):
        vals = group[col].values.astype(np.float32)
        diffs = np.subtract.outer(vals, vals)
        np.fill_diagonal(diffs, 0)
        return np.square(diffs).sum(axis=1)
    
    df['euclidean_mass'] = df.groupby('market_id').apply(lambda x: euclidean(x, mass_col)).explode().values
    df['euclidean_power'] = df.groupby('market_id').apply(lambda x: euclidean(x, power_col)).explode().values
    
    ## Local difference
    mass_thresh = threshold_std * df[mass_col].std()
    power_thresh = threshold_std * df[power_col].std()
    
    def local_diff(group, col, threshold):
        vals = group[col].values.astype(np.float32)
        diffs = np.abs(np.subtract.outer(vals, vals))
        np.fill_diagonal(diffs, 0)
        return (diffs < threshold).sum(axis=1)
    
    df['local_mass'] = df.groupby('market_id').apply(lambda x: local_diff(x, mass_col, mass_thresh)).explode().values
    df['local_power'] = df.groupby('market_id').apply(lambda x: local_diff(x, power_col, power_thresh)).explode().values
    
    return df

# Construct IVs
market_share_with_ivs = construct_differentiation_ivs(market_share, threshold_std=1)

# Display results
print(market_share_with_ivs)


Using columns: mass as mass, power as power


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20464\2981289498.py:45: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['euclidean_mass'] = df.groupby('market_id').apply(lambda x: euclidean(x, mass_col)).explode().values
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_20464\2981289498.py:46: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df['euclidean_power'] = df.groupby('market_id').apply(lambda x: euclidean(x, power_col

       year      type province brand     model fuel_type         mass  \
0      2019  国产新能源乘用车      上海市  东风风行     景逸S50       BEV  2036.000000   
1      2019  国产新能源乘用车      上海市    丰田       卡罗拉      PHEV  1975.000000   
2      2019  国产新能源乘用车      上海市    丰田        雷凌      PHEV  1975.000000   
3      2019  国产新能源乘用车      上海市    云度      云度π1       BEV  1785.000000   
4      2019  国产新能源乘用车      上海市    云度      云度π3       BEV  1845.000000   
...     ...       ...      ...   ...       ...       ...          ...   
79285  2023   国产燃油乘用车     黑龙江省    风神        皓极      PHEV  2093.000000   
79286  2023   国产燃油乘用车     黑龙江省   马自达  马自达CX-30       Gas  1880.750000   
79287  2023   国产燃油乘用车     黑龙江省   马自达   马自达CX-4       Gas  1977.363636   
79288  2023   国产燃油乘用车     黑龙江省   马自达   马自达CX-5       Gas  2018.479532   
79289  2023   国产燃油乘用车     黑龙江省   马自达   马自达CX-8       Gas  2441.315789   

            power  sales  weighted_Avg_Price  ...  product_id  market_id  \
0       90.000000      1                7.49  .

In [4]:
################ Construct exogenous cost-shifters

# Convert Year column into year in steel price data
steel_price['year'] = pd.to_datetime(steel_price['Year']).dt.year

# Drop 'Year' column as it's no longer needed
steel_price.drop(columns=['Year'], inplace=True)

# Rename columns for clarity
steel_price.rename(columns={'China: Steel Composite Price Indices:Annual:Average': 'steel_price_index'}, inplace=True)

# Use the mass of a model interacted with the steel price index as an exogenous cost-shifter
def construct_cost_shifters(market_share, steel_price):
    """
    Construct exogenous cost-shifters using weight and steel price index.
    
    Parameters:
    -----------
    market_share : DataFrame
        Must have a 'mass' column
    steel_price : DataFrame
        Must have a 'steel_price_index' column
    
    Returns:
    --------
    DataFrame with original data plus cost-shifter column
    """
    
    # Ensure steel price is aligned with market share data
    steel_price = steel_price.set_index('year')  # Assuming 'date' is the index
    
    # Merge on year
    merged = market_share.merge(steel_price, on='year', how='left')
    
    # Create cost-shifter as weight * steel price index
    merged['cost_shifter'] = merged['mass'] * merged['steel_price_index']
    
    return merged

# Construct cost-shifters
market_share_with_ivs_2 = construct_cost_shifters(market_share_with_ivs, steel_price)

In [163]:
################ Construct charging station IVs
# Construct cost-shifters
market_share_with_ivs_2 = construct_cost_shifters(market_share_with_ivs, steel_price)

In [164]:
################ Construct supply-side instrument for EV stock
# Import gas_price, gas_station_stock, road_length data
gas_price = pd.read_csv("C:/Users/Lenovo/Desktop/Dissertaion/China Data/Instruments/gas_price.csv")
road_length = pd.read_excel("C:/Users/Lenovo/Desktop/Dissertaion/China Data/Instruments/road_length.xlsx")
EV_stock = pd.read_excel("C:/Users/Lenovo/Desktop/Dissertaion/China Data/Demand/EV stock/2017-2022 EV_stock.xlsx")

In [165]:
################ Clean supply-side data
# Compute the yearly average of gas prices
gas_price_clean = gas_price[~gas_price['Name'].str.contains('Source', na=False)].copy()
gas_price_clean['year'] = pd.to_datetime(gas_price_clean['Name']).dt.year

# Create an average column for 92# and 95# gasoline prices
gas_price_clean['gas_price_avg'] = gas_price_clean[
    ["China: Market Price: Gasoline (92#): China VI", "China: Market Price: Gasoline (95#): China VI"]
].mean(axis=1)

# Calculate yearly average of gas price (using the average of 92# and 95#)
yearly_gas_price = (
    gas_price_clean
    .groupby('year')['gas_price_avg']
    .mean()
    .reset_index()
    .rename(columns={'gas_price_avg': 'yearly_gas_price_avg'})
)


In [166]:
# Export market share data with IVs and cost-shifters
market_share_with_ivs_2.to_csv('data_with_IV.csv', index=False)

In [183]:
# Generate supply-side data set directly from loaded supply-side sources (no merge with market_share_with_ivs_2)
province_mapping = {
    '北京': '北京市',
    '天津': '天津市',
    '河北': '河北省',
    '山西': '山西省',
    '内蒙古': '内蒙古自治区',
    '辽宁': '辽宁省',
    '吉林': '吉林省',
    '黑龙江': '黑龙江省',
    '上海': '上海市',
    '江苏': '江苏省',
    '浙江': '浙江省',
    '安徽': '安徽省',
    '福建': '福建省',
    '江西': '江西省',
    '山东': '山东省',
    '河南': '河南省',
    '湖北': '湖北省',
    '湖南': '湖南省',
    '广东': '广东省',
    '广西': '广西壮族自治区',
    '海南': '海南省',
    '重庆': '重庆市',
    '四川': '四川省',
    '贵州': '贵州省',
    '云南': '云南省',
    '西藏': '西藏自治区',
    '陕西': '陕西省',
    '甘肃': '甘肃省',
    '青海': '青海省',
    '宁夏': '宁夏回族自治区',
    '新疆': '新疆维吾尔自治区',
    '香港': '香港特别行政区' 
}

# Ensure province names are mapped for consistency
EV_stock_supply = EV_stock.copy()
EV_stock_supply['province'] = EV_stock_supply['province'].map(province_mapping)

road_length_supply = road_length.rename(columns={'地区': 'province'}).melt(
    id_vars='province', var_name='year', value_name='road_length'
)
road_length_supply['province'] = road_length_supply['province'].map(province_mapping)

# Calculate and add 2023 EV stock to EV_stock_supply

# Get 2022 EV stock for each province
stock_2022 = EV_stock_supply[EV_stock_supply['year'] == 2022][['province', 'EV_stock']].drop_duplicates().rename(columns={'EV_stock': 'EV_stock_2022'})

# Get 2023 province total EV sales (sum for electric vehicles) from market_share_with_ivs_2
sales_2023 = (
    market_share_with_ivs_2[
        (market_share_with_ivs_2['year'] == 2023) & (market_share_with_ivs_2['fuel_type'].isin(['BEV', 'PHEV']))
    ]
    .groupby('province', as_index=False)['sales']
    .sum()
)

# Merge 2022 stock and 2023 sales
merged = pd.merge(stock_2022, sales_2023, on='province', how='inner')
merged['EV_stock'] = merged['EV_stock_2022'] + merged['sales']
merged['year'] = 2023

# Select columns to match EV_stock_supply
ev_stock_2023 = merged[['province', 'year', 'EV_stock']]

# Append 2023 data to EV_stock_supply
EV_stock_supply = pd.concat([EV_stock_supply, ev_stock_2023], ignore_index=True)

# Combine all supply-side data
supply_side_data = (
    EV_stock_supply
    .merge(road_length_supply, on=['province', 'year'], how='left')
    .merge(yearly_gas_price, on='year', how='left')
)

# Calculate IVs
supply_side_data['road_fuel_IV'] = supply_side_data['road_length'] * supply_side_data['yearly_gas_price_avg']

# Calculate lagged EV stock for each province
supply_side_data = supply_side_data.sort_values(['province', 'year'])
supply_side_data['EV_stock_lag'] = supply_side_data.groupby('province')['EV_stock'].shift(1)

# Calculate lagged EV stock of other provinces
def lagged_other_province_stock(row, df):
    year = row['year']
    province = row['province']
    return df[(df['year'] == year) & (df['province'] != province)]['EV_stock_lag'].sum()

supply_side_data['other_province_EV_stock_lag'] = supply_side_data.apply(
    lambda row: lagged_other_province_stock(row, supply_side_data), axis=1
)
supply_side_data['shift_share_IV'] = supply_side_data['other_province_EV_stock_lag'] * supply_side_data['road_length']

# Add charging stations stock to supply-side data using a merge to ensure correct alignment
charging_stock = (
    market_share_with_ivs_2.groupby(['province', 'year'])['charging_stations_stock']
    .first()
    .reset_index()
)

supply_side_data = supply_side_data.merge(
    charging_stock, on=['province', 'year'], how='left'
)

# Select and export the columns you specified
cols = [
    'province', 'year', 'charging_stations_stock', 'EV_stock',
    'road_length', 'yearly_gas_price_avg', 'shift_share_IV', 'road_fuel_IV'
]

# Drop 2017 and 2018 data as they are not needed
supply_side_data = supply_side_data[~supply_side_data['year'].isin([2017, 2018])]

supply_side_data[cols].to_csv('supply_side_data.csv', index=False)